# Exercise 2 — apply_stop_loss

`apply_stop_loss` iterates over each bar, tracking the entry price. When the stop-loss is breached, it converts the signal from 1 to 0 and clears the entry so the next bar starts fresh. This stateful loop is the standard approach for stop-loss in a vectorised backtester.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def kelly_fraction(win_rate, avg_win, avg_loss):
    if avg_loss <= 0 or win_rate <= 0 or win_rate >= 1:
        return 0.0
    b = avg_win / avg_loss
    return max(0.0, min(1.0, win_rate - (1 - win_rate) / b))
def is_stopped_out(entry_price, current_price, stop_pct=0.05):
    if entry_price <= 0:
        return False
    return current_price <= entry_price * (1.0 - stop_pct)

def apply_stop_loss(signals, prices, stop_pct=0.05):
    """Apply stop-loss to a signal Series bar by bar.

    Algorithm:
      entry_price = None
      for i in range(len(result)):
          if result[i] == 1:
              if entry_price is None:       # new entry
                  entry_price = prices[i]
              elif is_stopped_out(entry_price, prices[i], stop_pct):
                  result[i] = 0             # exit position
                  entry_price = None        # clear entry
          else:
              entry_price = None            # position was already flat

    Returns pd.Series of {0, 1}.
    """
    # TODO: implement
    result = signals.copy().astype(float)
    # ... loop here ...
    return result.astype(int)


### Checks

In [ ]:
checks = 0

# 1 — returns same-length Series with {0,1} values
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    res = apply_stop_loss(sig, df["Close"])
    assert isinstance(res, pd.Series) and len(res) == len(df)
    assert set(res.unique()).issubset({0, 1})
    checks += 1; print("✅ 1 returns same-length Series with values in {0,1}")
except Exception as e:
    print("❌ 1:", e)

# 2 — stop-loss reduces long exposure (fewer or equal 1s)
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    res = apply_stop_loss(sig, df["Close"], stop_pct=0.05)
    assert res.sum() <= sig.sum(), "stop-loss should reduce or keep the same 1s"
    checks += 1; print("✅ 2 stop-loss does not increase long exposure")
except Exception as e:
    print("❌ 2:", e)

# 3 — controlled 5-bar test: stop hits at bar 3
try:
    dates = pd.date_range("2023-01-01", periods=5, freq="B")
    prices = pd.Series([100.0, 102.0, 98.0, 93.0, 95.0], index=dates)
    sig    = pd.Series([1, 1, 1, 1, 1], index=dates)
    res    = apply_stop_loss(sig, prices, stop_pct=0.05)
    expected = [1, 1, 1, 0, 1]
    assert res.tolist() == expected, f"expected {expected}, got {res.tolist()}"
    checks += 1; print("✅ 3 stop hits at bar 3 (93 ≤ 100×0.95=95), re-enters bar 4")
except Exception as e:
    print("❌ 3:", e)

# 4 — all-rising prices: no stop ever triggers
try:
    dates   = pd.date_range("2023-01-01", periods=20, freq="B")
    prices  = pd.Series([float(100 + i) for i in range(20)], index=dates)
    sig     = pd.Series(1, index=dates)
    res     = apply_stop_loss(sig, prices, stop_pct=0.05)
    assert (res == 1).all(), "rising prices → no stop-loss, all signals stay 1"
    checks += 1; print("✅ 4 all-rising prices → no stop triggers → all signals stay 1")
except Exception as e:
    print("❌ 4:", e)

# 5 — already-flat signals pass through unchanged
try:
    dates  = pd.date_range("2023-01-01", periods=5, freq="B")
    prices = pd.Series([100.0, 80.0, 60.0, 40.0, 20.0], index=dates)
    sig    = pd.Series([0, 0, 0, 0, 0], index=dates)
    res    = apply_stop_loss(sig, prices, stop_pct=0.05)
    assert (res == 0).all(), "flat signals should be unchanged"
    checks += 1; print("✅ 5 flat signals (all 0) pass through unchanged")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
